In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import QuantileTransformer
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:

# ------------------------- Basisverzeichnis -------------------------
base_dir = Path.cwd()

# ------------------------- Pfad zur imputierten Datenbank (Output 4.1) -------------------------
# Wir gehen davon aus, dass 4.1 parallel zu 4.2 liegt
# ../4.1_Imputation/Imputed_Daten_Raw.csv
input_path = base_dir.parent / "4.1_Imputation" / "Imputed_Data_Raw.csv"

print("Verwendeter Datenbankpfad:")
print(input_path)

if not input_path.exists():
    raise FileNotFoundError(f"Imputation-File nicht gefunden: {input_path}")

df = pd.read_csv(input_path, low_memory=False)
print(f"Datensatz geladen mit {df.shape[0]} Zeilen und {df.shape[1]} Spalten.")

Verwendeter Datenbankpfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.1_Imputation\Imputed_Data_Raw.csv


Datensatz geladen mit 175099 Zeilen und 32 Spalten.


<div class="alert alert-info">
    <b>4.2 Vorverarbeitung (Imputed Data)</b><br>
    Adaptiert von 3.1 Vorverarbeitung.
    <br>
    <b>1. Log-Transformation</b> (np.log1p)<br>
    <b>2. Robust Scaling</b> (QuantileTransformer)<br>
</div>

In [4]:
# ------------------------- Ausgabeverzeichnis -------------------------
output_dir = base_dir
# (Kein Zeitstempel-Unterordner nötig, da wir überschreiben oder direkt nutzen wollen in 4.3)
# Oder doch Zeitstempel? Pipeline-Standard ist oft Zeitstempel.
# Wir machen es einfach: Direkt in den Ordner für den einfachen Zugriff in 4.3
print(f"Output Ordner: {output_dir}")

Output Ordner: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.2_Preprocessing


In [5]:
# ------------------------- Spaltenausschluss -------------------------
EXCLUDE_COLS = [
    "WGS84_latitude",
    "WGS84_longitude",
    "Database_number",
    "id",                 
    "index",
    "temperature_in_c",
    "rock_type",
    "IBE", "IBE_Calculated" # IBE nicht transformieren
]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
transform_candidates = [c for c in numeric_cols if c not in EXCLUDE_COLS]

print(f"{len(transform_candidates)} numerische Spalten sollten transformiert werden")

19 numerische Spalten sollten transformiert werden


In [6]:
# ------------------------- Log-Transformation -------------------------
skewness = df[transform_candidates].skew().sort_values(ascending=False)
skewed_cols = skewness[abs(skewness) > 1.0].index.tolist()

df_base = df.copy()
base_cols = []

pdf_log_path = output_dir / "Log_Transformation_Report.pdf"

with PdfPages(pdf_log_path) as pdf:
    fig = plt.figure(figsize=(10, 6))
    plt.text(0.5, 0.5, f"Log-Transformation Report (Imputed Data)", ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig(fig)
    plt.close()

    for col in transform_candidates:
        # Prüfen auf negative Werte für log1p (sollten in Raw geochemie nicht vorkommen, aber sicher ist sicher)
        if df[col].min() >= 0 and col in skewed_cols:
            new_col = f"{col}_log"
            df_base[new_col] = np.log1p(df[col])
            base_cols.append(new_col)
            
# ------------------------- Visualisierung -------------------------
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.histplot(df[col].dropna(), bins=30, ax=axes[0], color="skyblue", kde=True)
            axes[0].set_title(f"Orig: {col}")
            sns.histplot(df_base[new_col].dropna(), bins=30, ax=axes[1], color="lightgreen", kde=True)
            axes[1].set_title(f"Transformed: ln(1 + {col})")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
        else:
            base_cols.append(col)
print(f"Log-Report erstellt: {pdf_log_path}")

Log-Report erstellt: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.2_Preprocessing\Log_Transformation_Report.pdf


In [7]:
# ------------------------- Gaussian Scaling (QuantileTransformer) -------------------------
print('STARTING GAUSSIAN SCALING EXECUTION')
df_final = df_base.copy()
final_features = []

pdf_report_path = output_dir / "Gaussian_Scaling_Report.pdf"

with PdfPages(pdf_report_path) as pdf:
    fig = plt.figure(figsize=(10, 6))
    plt.text(0.5, 0.5, f"Gaussian Scaling Report (Imputed Data)", ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig(fig)
    plt.close()

    for col in base_cols:
        full_series = df_base[col]
        valid_mask = full_series.notna()
        
        if valid_mask.sum() > 0:
            scaler_col = QuantileTransformer(output_distribution='normal', random_state=42)
            # Reshape für SKLearn
            vals = full_series.dropna().values.reshape(-1, 1)
            final_trans = scaler_col.fit_transform(vals).flatten()
            
            new_col_name = f"{col}_gauss"
            df_final.loc[valid_mask, new_col_name] = final_trans
            final_features.append(new_col_name)
            
# ------------------------- Visualisierung -------------------------
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.histplot(full_series, bins=30, ax=axes[0], color="lightgreen", kde=True)
            axes[0].set_title(f"Input: {col}")
            sns.histplot(final_trans, bins=30, ax=axes[1], color="crimson", kde=True)
            axes[1].set_title(f"Output: Gaussian Scaled")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

print(f"Gaussian Report: {pdf_report_path}")

STARTING GAUSSIAN SCALING EXECUTION


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (570). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (330). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (427). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (20). n_quantiles is set to n_samples.
  warnings.warn(


Gaussian Report: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.2_Preprocessing\Gaussian_Scaling_Report.pdf


In [8]:
# ------------------------- Speichern -------------------------
output_file_som = output_dir / "Preprocessed_SOM_Ready.csv"

# Nur relevante Spalten speichern
final_cols_to_save = EXCLUDE_COLS + final_features
final_cols_real = [c for c in final_cols_to_save if c in df_final.columns]

df_final[final_cols_real].to_csv(output_file_som, index=False)
print(f"SOM Data gespeichert: {output_file_som}")

SOM Data gespeichert: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\4_Imputation\4.2_Preprocessing\Preprocessed_SOM_Ready.csv
